# 02. Data Preprocessing


In [1]:
import pandas as pd
import pathlib

RAW_DATA_PATH = pathlib.Path("data/raw")
PROCESSED_DATA_PATH = pathlib.Path("data/processed")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

csv_files = sorted(list(RAW_DATA_PATH.glob("*.csv")))

# 2. Robust Parsing Functions
The SJPD dataset has inconsistent timestamp formats across years. We define custom functions to handle this complexity.

In [ ]:
def parse_cdts_series(raw_series: pd.Series) -> pd.Series:
    # Normalize raw values and remove known suffix noise.
    s = (
        raw_series.astype("string")
        .str.upper()
        .str.replace("PS", "", regex=False)
        .str.strip()
    )

    # Primary parse: first 14 digits interpreted as YYYYMMDDHHMMSS.
    digits14 = s.str.extract(r"(\d{14})", expand=False)
    parsed = pd.to_datetime(digits14, format="%Y%m%d%H%M%S", errors="coerce")

    # Secondary parse: 12-digit timestamps (YYYYMMDDHHMM) -> append seconds.
    missing_mask = parsed.isna()
    if missing_mask.any():
        digits12 = s[missing_mask].str.extract(r"(\d{12})", expand=False)
        parsed12 = pd.to_datetime(
            digits12 + "00", format="%Y%m%d%H%M%S", errors="coerce"
        )
        parsed.loc[missing_mask] = parsed12.values

    # Final fallback: let pandas infer any remaining odd formats.
    missing_mask = parsed.isna()
    if missing_mask.any():
        parsed_fallback = pd.to_datetime(s[missing_mask], errors="coerce")
        parsed.loc[missing_mask] = parsed_fallback.values

    return parsed


def check_data_quality(df: pd.DataFrame, file_name: str):
    """Checks for and prints problematic rows in the dataframe."""
    print(f"--- Data Quality Audit for {file_name} ---")

    # Check for missing timestamps
    missing_cdts = df["CDTS"].isna().sum()
    if missing_cdts > 0:
        print(f"[WARNING] Found {missing_cdts} rows with missing CDTS (timestamps).")

    # Check for missing priority
    if "PRIORITY" in df.columns:
        missing_priority = df["PRIORITY"].isna().sum()
        if missing_priority > 0:
            print(f"[WARNING] Found {missing_priority} rows with missing PRIORITY.")
    else:
        print(f"[ERROR] 'PRIORITY' column missing entirely!")

    # Sample problematic rows
    problematic = df[
        df["CDTS"].isna()
        | (df["PRIORITY"].isna() if "PRIORITY" in df.columns else False)
    ]
    if not problematic.empty:
        print("Sample problematic rows:")
        # Use print head instead of display for simplicity in script-based check if needed,
        # but for notebook it's fine to keep 'display' if the environment supports it.
        print(problematic.head(3))
    else:
        print("No immediate data quality issues detected.")
    print("------------------------------------------\n")

# 3. Sequential File Processing
We load each file, apply our cleaning logic, and flag important features like 'Canceled' calls and 'Priority 1' calls.

In [3]:
processed_dfs = []
for file_path in csv_files:
    print(f"Processing {file_path.name}...")
    df = pd.read_csv(file_path, low_memory=False)
    check_data_quality(df, file_path.name)
    df["CDTS_RAW"] = df["CDTS"].astype("string")
    df["CDTS"] = parse_cdts_series(df["CDTS"])
    df["year"] = df["CDTS"].dt.year
    df["month"] = df["CDTS"].dt.month
    df["hour"] = df["CDTS"].dt.hour
    df["day_of_week"] = df["CDTS"].dt.day_name()
    df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"])
    df["time_bin"] = df["CDTS"].dt.floor("15min")
    if "FINAL_DISPO" in df.columns:
        df["is_canceled"] = df["FINAL_DISPO"].str.contains("CAN", case=False, na=False)
    else:
        df["is_canceled"] = False
    if "PRIORITY" in df.columns:
        df["PRIORITY_NUM"] = pd.to_numeric(df["PRIORITY"], errors="coerce")
        df["is_priority_1"] = (df["PRIORITY_NUM"] == 1)
    else:
        df["is_priority_1"] = False
    processed_dfs.append(df)
master_df = pd.concat(processed_dfs, ignore_index=True)

Processing policecalls2016.csv...


--- Data Quality Audit for policecalls2016.csv ---
------------------------------------------



Processing policecalls2017.csv...


--- Data Quality Audit for policecalls2017.csv ---
------------------------------------------



Processing policecalls2018.csv...


...

--- Data Quality Audit for policecalls2024.csv ---
------------------------------------------



Processing policecalls2025.csv...


--- Data Quality Audit for policecalls2025.csv ---
------------------------------------------



Processing policecalls2026.csv...


--- Data Quality Audit for policecalls2026.csv ---
------------------------------------------



# 5. Parquet Data Export
We save the combined file to the `data/processed` directory.

In [4]:
min_year = master_df["year"].min()
max_year = master_df["year"].max()
out_path = PROCESSED_DATA_PATH / f"sjpd_calls_{min_year}_{max_year}_processed.parquet"
master_df.to_parquet(out_path, index=False)
print(f"Saved cleaned dataset to: {out_path}")

Saved cleaned dataset to: data\processed\sjpd_calls_2016_2026_processed.parquet
